# S18 · Black-Scholes: price the Reliance option

Our option: the right to buy Reliance at **₹1,560** within one month of 31 December
2025. What is that right fairly worth today?

The key sentence: **price = discounted average future payoff**. Black-Scholes works that
average out exactly. Think of it as a calculator: five inputs in, a call price and a put
price out. Four inputs we can read off a screen. The fifth, **volatility**, we measure
from Reliance's real 2025 prices. Then we turn the volatility dial, read the Greeks, and
watch time value melt away as expiry comes closer.


**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press the play button on each
  cell, top to bottom, and read the plain-English note above each one.
- New to the words (call, put, strike, premium)? Each one is explained the moment it
  appears, and all of them are in `primers/glossary.md`.
- New to the bell curve? `primers/distributions_and_the_bell_curve.md` is a ten-minute,
  picture-first read.
- Already confident with code or with finance? Look for the cells marked
  **Stretch (optional)** near the end.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook uses numpy, pandas, matplotlib and scipy, which Colab already ships.
# So there is nothing to install here.
print("Setup complete - nothing to install.")

## Step 1 — get our tools ready

Besides NumPy, pandas and matplotlib, we need `norm` from scipy: the standard bell curve.
`norm.cdf(x)` is the area under the bell curve to the left of `x`, which the formula uses.

In [ ]:
from pathlib import Path
import numpy as np                 # fast maths on lists of numbers
import pandas as pd                # tables of data
import matplotlib.pyplot as plt    # charts
from scipy.stats import norm       # the bell curve

print("Tools ready.")

## Step 2 — load Reliance's prices

These are Reliance's daily closing prices from January 2023 to December 2025, the same NSE
data we used in S16 and S17. They were downloaded once with
`yf.download("RELIANCE.NS", start="2023-01-01", end="2026-01-01", auto_adjust=True)` and
saved, so everyone gets exactly the numbers on the slides, with or without internet.

In [ ]:
# Where the saved prices live: next to this notebook, or on the course website.
local_copy = Path("../data/reliance_2023_2025.csv")
website_copy = ("https://girishmkulkarni.github.io/applied-maths-in-industry-site/"
                "sessions/S18/data/reliance_2023_2025.csv")

if local_copy.exists():
    prices = pd.read_csv(local_copy, index_col=0, parse_dates=True)
else:
    prices = pd.read_csv(website_copy, index_col=0, parse_dates=True)

print("days of prices:", len(prices))
print("from", prices.index[0].date(), "to", prices.index[-1].date())
prices.tail()

## Step 3 — how jumpy is Reliance? Measuring volatility

**Volatility**, written σ (sigma), measures how much a price swings. We estimate it from
history in two moves:

1. Take the daily percentage changes during 2025 and find their standard deviation: the
   typical daily move.
2. Scale it up to a year by multiplying by √252, because a year has about 252 trading days
   and uncertainty grows with the **square root** of time, not with time itself.

In [ ]:
# Daily % change of Reliance's price during 2025.
daily = prices["Close"].loc["2025"].pct_change().dropna()

daily_swing = daily.std()                  # typical daily move
sigma = daily_swing * np.sqrt(252)         # scaled up to a year

print("daily changes in 2025     :", len(daily))
print("typical daily move        :", round(100 * daily_swing, 2), "%")
print("volatility, sigma, a year :", round(100 * sigma, 1), "%")

## Step 4 — the calculator

Below is the Black-Scholes formula written once as a Python function, plus a second
function for the Greeks. **You do not need to read inside them.** Run the cell, then use
them like a calculator. (The Stretch section at the end opens the first one up.)

In [ ]:
def black_scholes_price(S, K, T, r, sigma, kind="call"):
    """Fair price today of a European call or put (the Black-Scholes formula).

    S     : stock price today
    K     : strike price
    T     : time to expiry, in years (one month = 1/12)
    r     : interest rate, per year (6% = 0.06)
    sigma : volatility, per year (19.5% = 0.195)
    kind  : "call" or "put"
    """
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if kind == "call":
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)


def greeks(S, K, T, r, sigma, kind="call"):
    """How much the option price moves when one input moves a little.

    Same inputs as black_scholes_price. Returns a dictionary:
      delta         : change in price for a Rs 1 move in the stock
      gamma         : change in delta for a Rs 1 move in the stock
      theta_per_day : change in price after one more day passes
      vega_per_1pct : change in price if volatility rises by 1 percentage point
      rho_per_1pct  : change in price if the interest rate rises by 1 percentage point
    """
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    # Gamma and Vega are the same for a call and a put.
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    vega = S * norm.pdf(d1) * np.sqrt(T)

    if kind == "call":
        delta = norm.cdf(d1)
        theta = -S * norm.pdf(d1) * sigma / (2 * np.sqrt(T)) - r * K * np.exp(-r * T) * norm.cdf(d2)
        rho = K * T * np.exp(-r * T) * norm.cdf(d2)
    else:
        delta = norm.cdf(d1) - 1
        theta = -S * norm.pdf(d1) * sigma / (2 * np.sqrt(T)) + r * K * np.exp(-r * T) * norm.cdf(-d2)
        rho = -K * T * np.exp(-r * T) * norm.cdf(-d2)

    return {"delta": delta, "gamma": gamma, "theta_per_day": theta / 365,
            "vega_per_1pct": vega / 100, "rho_per_1pct": rho / 100}


print("Calculator ready: black_scholes_price(...) and greeks(...) are defined.")

## Step 5 — press the button

The five inputs:

| Input | Meaning | Our value |
|---|---|---|
| `S` | Reliance's price today | the last price in our data |
| `K` | strike | ₹1,560 |
| `T` | time to expiry, in years | one month = 1/12 |
| `r` | interest rate on safe money | 6% a year |
| `sigma` | volatility | from Step 3 |

In [ ]:
S = prices["Close"].iloc[-1]          # price on 31 Dec 2025
K = 1560                              # strike
T = 1 / 12                            # one month, in years
r = 0.06                              # 6% a year

call = black_scholes_price(S, K, T, r, sigma, kind="call")
put = black_scholes_price(S, K, T, r, sigma, kind="put")

print("S =", round(S, 2), "  sigma =", round(sigma, 3))
print("call price:", round(call, 2), "  put price:", round(put, 2))

The right to buy Reliance at ₹1,560 for a month is worth about ₹40.81 a share today, and
the right to sell at ₹1,560 about ₹29.86.

## Step 6 — where does the price come from? Intrinsic value and time value

If you used the call right now you would gain only S − K, about ₹3. The rest of the price is
**time value**: the chance that Reliance climbs further before expiry.

In [ ]:
intrinsic = max(S - K, 0)        # what using the call today would pay
time_value = call - intrinsic

print("call price     :", round(call, 2))
print("intrinsic value:", round(intrinsic, 2))
print("time value     :", round(time_value, 2))

## Step 7 — a free sanity check: put-call parity

From notebook 1: `C + K e^(-rT)` must equal `P + S`. If our calculator is right, the two sides
match.

In [ ]:
left = call + K * np.exp(-r * T)
right = put + S

print("C + K*e^(-rT):", round(left, 4))
print("P + S        :", round(right, 4))
print("difference   :", round(left - right, 10))

## Step 8 — which option is worth more? Calm versus jumpy, by hand

Before the real thing, a toy. Two stocks at ₹1,000, and a call on each with strike ₹1,000.
Stock A is calm: in a month it ends at ₹990 or ₹1,010, half and half. Stock B is jumpy: it
ends at ₹900 or ₹1,100. Which call is worth more?

In [ ]:
toy_strike = 1000
calm_endings = np.array([990, 1010])      # stock A moves 1%
jumpy_endings = np.array([900, 1100])     # stock B moves 10%

calm_payoffs = np.maximum(calm_endings - toy_strike, 0)
jumpy_payoffs = np.maximum(jumpy_endings - toy_strike, 0)

print("calm stock : payoffs", calm_payoffs, "  average payoff", calm_payoffs.mean())
print("jumpy stock: payoffs", jumpy_payoffs, "  average payoff", jumpy_payoffs.mean())

In a bad month both calls pay zero, because the buyer simply walks away however far the
stock falls. In a good month the jumpy call pays ten times more. Price is the average payoff,
so **more uncertainty means more option value**.

## Step 9 — turn only the volatility dial

Now with the real Reliance option. Keep S, K, T and r fixed and change only sigma.

In [ ]:
for vol in [0.10, 0.20, 0.30, 0.40]:
    price = black_scholes_price(S, K, T, r, vol, kind="call")
    print("volatility", vol, " ->  call price", round(price, 2))

The same thing as a picture, over many volatilities, for the call and the put.

In [ ]:
vol_grid = np.linspace(0.05, 0.60, 50)

call_by_vol = [black_scholes_price(S, K, T, r, v, kind="call") for v in vol_grid]
put_by_vol = [black_scholes_price(S, K, T, r, v, kind="put") for v in vol_grid]

plt.figure(figsize=(8, 5))
plt.plot(100 * vol_grid, call_by_vol, color="#2E75B6", linewidth=3, label="call")
plt.plot(100 * vol_grid, put_by_vol, color="#C0392B", linewidth=3, label="put")
plt.axvline(100 * sigma, color="grey", linestyle=":", label="Reliance, 2025")
plt.xlabel("volatility, % a year")
plt.ylabel("option price (Rs)")
plt.title("A jumpier stock makes both options worth more")
plt.legend()
plt.show()

## Step 10 — the Greeks of our call

The **Greeks** say how much the price moves when one input moves a little:

- **Delta:** change in the option price when Reliance moves ₹1.
- **Gamma:** change in Delta when Reliance moves ₹1.
- **Theta:** change in the price after one more day passes (time decay).
- **Vega:** change in the price if volatility rises by 1 percentage point.
- **Rho:** change if the interest rate rises by 1 percentage point. Small here; just know the name.

In [ ]:
g = greeks(S, K, T, r, sigma, kind="call")

print("Delta:", round(g["delta"], 2))
print("Gamma:", round(g["gamma"], 4))
print("Theta:", round(g["theta_per_day"], 2), "rupees a day")
print("Vega :", round(g["vega_per_1pct"], 2), "rupees per 1% more volatility")
print("Rho  :", round(g["rho_per_1pct"], 2), "rupees per 1% higher interest rate")

Read them as sentences. Reliance up ₹10: the call gains about ₹5.60. A quiet day with no
news: the call loses about 71 paise. Volatility up from 19.5% to 20.5%: the call gains about
₹1.78.

## Step 11 — check that Delta really is "change per ₹1"

Nudge Reliance up by ₹1, re-price, and compare the change with Delta.

In [ ]:
price_now = black_scholes_price(S, K, T, r, sigma, kind="call")
price_up = black_scholes_price(S + 1, K, T, r, sigma, kind="call")

print("call now              :", round(price_now, 3))
print("call if Reliance +Rs 1:", round(price_up, 3))
print("change                :", round(price_up - price_now, 3))
print("Delta said            :", round(g["delta"], 3))

## Step 12 — watch time value melt (Theta)

Keep everything fixed except the days left. As expiry comes closer, the price curve sinks
onto the hockey stick from notebook 1: the time value melts away.

In [ ]:
for days_left in [30, 20, 10, 5, 1]:
    price = black_scholes_price(S, K, days_left / 365, r, sigma, kind="call")
    print(days_left, "days left  ->  call price", round(price, 2))

spot_grid = np.linspace(1400, 1720, 200)
plt.figure(figsize=(8, 5))
for days_left, colour in [(30, "#2E75B6"), (10, "#E08E0B"), (1, "#6A4C93")]:
    curve = [black_scholes_price(s, K, days_left / 365, r, sigma) for s in spot_grid]
    plt.plot(spot_grid, curve, color=colour, linewidth=2.5, label=str(days_left) + " days left")
plt.plot(spot_grid, np.maximum(spot_grid - K, 0), color="grey", linestyle="--", label="payoff at expiry")
plt.axvline(K, color="grey", linestyle=":")
plt.xlabel("Reliance price now (Rs)")
plt.ylabel("call price (Rs)")
plt.title("As expiry nears, the price sinks onto the hockey stick")
plt.legend()
plt.show()

### Stretch (optional) — open the calculator

Skip this if you are new to maths. The formula is

$$ C = S\,N(d_1) - K e^{-rT} N(d_2), \qquad
d_1 = \frac{\ln(S/K) + (r + \sigma^2/2)\,T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T} $$

where `N` is `norm.cdf`. Read it as: the value of the shares you expect to get, minus today's
value of the cash you expect to pay. `N(d2)` is roughly the chance that the call ends in the
money. We do not derive it here; that needs stochastic calculus.

In [ ]:
d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
d2 = d1 - sigma * np.sqrt(T)

shares_part = S * norm.cdf(d1)
cash_part = K * np.exp(-r * T) * norm.cdf(d2)

print("d1 =", round(d1, 4), "   d2 =", round(d2, 4))
print("N(d2), roughly the chance of ending in the money:", round(norm.cdf(d2), 3))
print("shares part S*N(d1)       :", round(shares_part, 2))
print("cash part K*e^(-rT)*N(d2) :", round(cash_part, 2))
print("call = shares - cash      :", round(shares_part - cash_part, 2))

## Your turn

1. Price the same Reliance call and put with **three months** to expiry instead of one
   (`T = 3 / 12`). Which is worth more, and why?
2. Price a call with a higher strike, **₹1,600**, for one month. Is it cheaper or dearer than
   the ₹1,560 call? Use the payoff picture from notebook 1 to explain.

In [ ]:
# Your code here: two new prices for question 1, one for question 2.

## What you just did

You measured Reliance's volatility from a year of real prices (19.5% a year), used the
Black-Scholes calculator to price the one-month ₹1,560 call (₹40.81) and put (₹29.86), and
split the call's price into ₹3 of intrinsic value and ₹38 of time value. You saw why a
jumpier stock makes options worth more, read the Greeks as everyday sentences, and watched
time value melt as expiry came closer.

We used a formula we did not derive. Notebook 3 checks it a completely different way: by
simulating Reliance's next month 100,000 times.